# GovAI: стриминговый инференс на Kaggle

Этот ноутбук демонстрирует **базовый пример вызова модели с потоковой генерацией** (streaming). Здесь используется только сама языковая модель — **без RAG, без внешних инструментов и без сложной оркестрации**. Подходит для быстрой проверки качества ответов обученной (`GovAI Qwen3.5-4B LoRA`) и базовой (`Qwen3.5-4B`) моделей на одном и том же промпте.

In [3]:
# Промпты (системный и пользовательский)
SYSTEM = (
    "You are a specialized analyst for government organizations. "
    "Analyze budgets by KOSGU, procurement according to 44-FZ, "
    "develop development strategies for budget directions. "
    "Respond in Russian with structured analysis."
)

prompt = "Проанализируй бюджет на закупку оборудования (КОСГУ 310) 10 млн руб. Риски по 44-ФЗ при сокращении 15%?"

In [4]:
"""
Инференс обученной модели GovAI Qwen3.5-4B с поддержкой стриминга
"""

!pip install -q unsloth
from unsloth import FastLanguageModel
from transformers import TextIteratorStreamer
import torch
import threading

LORA_OUTPUT = "/kaggle/input/models/startexe/govai-qwen-lora-v11/other/default/1"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT,
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)


def stream_generate(prompt: str, max_new_tokens: int = 1024):
    """Генерация с потоковым выводом токенов."""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [{"type": "text", "text": prompt}]},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    attention_mask = torch.ones_like(inputs)

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    # Генерация в отдельном потоке (streamer блокирует до первого токена)
    generation_kwargs = dict(
        input_ids=inputs,
        attention_mask=attention_mask,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=1.0,
        top_k=20,
        top_p=0.95,
        min_p=0.0,
        repetition_penalty=1.0,
        do_sample=True,
    )

    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    for text in streamer:
        yield text

    thread.join()


if __name__ == "__main__":
    print("=" * 60)
    print("ОБУЧЕННАЯ МОДЕЛЬ (GovAI LoRA) — Streaming output:")
    print("-" * 40)
    for token_text in stream_generate(prompt, max_new_tokens=4096):
        print(token_text, end="", flush=True)
    print("\n" + "-" * 40)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 25.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.1 MB/s eta 0:00

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

ОБУЧЕННАЯ МОДЕЛЬ (GovAI LoRA) — Streaming output:
----------------------------------------
The user is asking me to analyze a budget allocation for equipment procurement (KOSGU 310) of 10 million rubles, and identify risks according to 44-FZ if there's a 15% reduction in this budget.

I need to respond in Russian with structured analysis, following the KOSGU classification system and 44-FZ (Government Procurement Law of Russia) regulations.

Key points to cover:
1. KOSGU 310 explanation (Services for equipment)
2. Budget analysis
3. Risk factors in procurement under 44-FZ
4. Recommendations

Let me create a structured analysis with professional language appropriate for government context.
</think>

## Бюджетная закупка: Оборудование (310)

**Бюджет:** 10 млн руб.
**Риски при сокращении:** 15%

### Классификатор КОСГУ

- **310**: Услуги по обеспечению безопасности и охраны

### Бюджетный анализ

- **Исходный бюджет:** 10 000 000 руб.
- **После сокращения:** 8 500 000 руб. (списание 15%)

In [5]:
"""
Инференс НЕобученной модели Qwen3.5-4B с поддержкой стриминга
"""

!pip install -q unsloth
from unsloth import FastLanguageModel
from transformers import TextIteratorStreamer
import torch
import threading

# БАЗОВАЯ модель, без LoRA
BASE_MODEL = "unsloth/Qwen3.5-4B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)


def stream_generate(prompt: str, max_new_tokens: int = 1024):
    """Генерация с потоковым выводом токенов."""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [{"type": "text", "text": prompt}]},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    attention_mask = torch.ones_like(inputs)

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    generation_kwargs = dict(
        input_ids=inputs,
        attention_mask=attention_mask,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=1.0,
        top_k=20,
        top_p=0.95,
        min_p=0.0,
        repetition_penalty=1.0,
        do_sample=True,
    )

    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    for text in streamer:
        yield text

    thread.join()


if __name__ == "__main__":
    print("=" * 60)
    print("БАЗОВАЯ МОДЕЛЬ (без LoRA) — Streaming output:")
    print("-" * 40)
    for token_text in stream_generate(prompt, max_new_tokens=4096):
        print(token_text, end="", flush=True)
    print("\n" + "-" * 40)


==((====))==  Unsloth 2026.6.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

БАЗОВАЯ МОДЕЛЬ (без LoRA) — Streaming output:
----------------------------------------
Here's a.generation of the thought process to lead to the suggested response:

1.  **Analyze the Request:**
    *   **Role:** Specialist for government organizations (budget/procurement analysis).
    *   **Task:** Analyze a budget for equipment purchase (KOSGU 310) amounting to 10 million RUB with a potential 15% reduction (due to a 44-FZ procurement process).
    *   **Specifics:** Focus on 44-FZ (Federal Law No. 223-FZ and 44-FZ are key, but specifically 44-FZ for procurement), KOSGU, and risk management associated with a 15% cut.
    *   **Language:** Russian.
    *   **Format:** Structured analysis.

2.  **Understand the Context & Constraints:**
    *   **Budget:** 10 million RUB for equipment (310 - Equipment for material-technical needs).
    *   **Change:** 15% reduction (i.e., the budget becomes 8.5 million RUB, or the *expected* spend is 8.5M, or the contract is for 8.5M but budgeted for 10